[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/01_autograd/01_autograd.ipynb)

# 01 · 自动微分内核（从零写 autograd）

目标：把 `loss.backward()` 从零写出来。先写 **micrograd 式标量引擎**（`Value`），再写 **张量版 autograd**（含**广播梯度**），全程用 **数值梯度对拍**，最后用它**训通一个小 MLP**。

路线：标量 `Value` 引擎 → 拓扑逆序 backward → 数值梯度对拍 → 张量 VJP + `unbroadcast` → 训练 MLP → ✏️ 练习（标量算子 / broadcast 求和 / 自定义 op）→ 📖 答案 → 🧪 真实数据胶囊（对照 torch.autograd）。

> 心智模型：**autograd = 一张计算图 + 每个算子的 VJP 规则 + 拓扑逆序遍历**。没有魔法。

## 1 · micrograd 式标量引擎 `Value`

每个 `Value` 携带三样：`data`（数值）、`grad`（loss 对它的梯度，初始 0）、`_backward`（把自己的梯度分发给父节点的闭包）。
每个运算：算出 `out.data`，并给 `out` 挂一个 `_backward`。**注意梯度用 `+=` 累加**（一个节点可能被多处使用）。

In [ ]:
import numpy as np
import math
rng = np.random.default_rng(0)

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None
    def __repr__(self):
        return f'Value(data={self.data:.4f}, grad={self.grad:.4f})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += out.grad      # d(a+b)/da = 1
            other.grad += out.grad      # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad   # d(ab)/da = b
            other.grad += self.data  * out.grad   # d(ab)/db = a
        out._backward = _backward
        return out

    def __pow__(self, k):               # 只支持常数幂
        assert isinstance(k, (int, float))
        out = Value(self.data ** k, (self,), f'**{k}')
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t * t) * out.grad   # d tanh/dx = 1 - tanh^2
        out._backward = _backward
        return out

    # —— 便利运算（由上面几个基本算子组合）——
    def __neg__(self):        return self * -1
    def __sub__(self, o):     return self + (-o if isinstance(o, Value) else Value(-o))
    def __radd__(self, o):    return self + o
    def __rmul__(self, o):    return self * o

a = Value(2.0); b = Value(-3.0)
c = a * b + a ** 2          # = 2*-3 + 4 = -2
print('前向:', c)
assert abs(c.data - (-2.0)) < 1e-12
print('✅ Value 前向正确；每个 out 都挂好了 _backward 闭包')

## 2 · 拓扑逆序 `backward()`

反向：从 loss 节点出发，按计算图**拓扑逆序**依次调用每个节点的 `_backward`。
拓扑逆序保证：轮到一个节点时它所有下游梯度已累加完，于是能拿到完整的 `out.grad` 再分发给父节点。
起点 `loss.grad = 1.0`（loss 对自己的导数）。

In [ ]:
def backward(root):
    '''对 root（通常是标量 loss）做反向传播，填好图里每个节点的 .grad。'''
    topo = []
    visited = set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            topo.append(v)          # 后序 -> topo 是拓扑序（父在子后）
    build(root)
    root.grad = 1.0                  # dloss/dloss = 1
    for v in reversed(topo):        # 拓扑逆序
        v._backward()
    return topo

# 测一个标量函数 f(a,b) = (a*b + a^2) ; 手算梯度: df/da = b + 2a, df/db = a
a = Value(2.0); b = Value(-3.0)
f = a * b + a ** 2
backward(f)
print(f'da = {a.grad}  (手算 b+2a = {-3 + 2*2})')
print(f'db = {b.grad}  (手算 a   = {2.0})')
assert abs(a.grad - (-3 + 4)) < 1e-12
assert abs(b.grad - 2.0) < 1e-12
print('✅ 拓扑逆序 backward 正确')

**验证梯度累加**：让同一个节点被用两次（`a` 出现在 `a*b` 和 `a**2` 两处），梯度应是两条路径之和。
若 `_backward` 误用 `=` 而非 `+=`，下面会算错。

In [ ]:
# d/da [ a*b + a*a ] = b + 2a  —— a 走了两条路径，梯度相加
a = Value(5.0); b = Value(7.0)
f = a * b + a * a
backward(f)
assert abs(a.grad - (7 + 2*5)) < 1e-12, '梯度必须沿两条路径累加 (+=)'
print(f'da = {a.grad} == b+2a = {7+10} ✅  (证明 += 累加生效，分叉节点梯度求和)')

## 3 · 用数值梯度对拍（autograd 的黄金参考）

怎么确信 `backward()` 没写错？用**数值梯度**（中心差分）对拍——它与我们的实现无关。
对一个稍复杂的标量函数，逐个输入比较解析梯度 vs 数值梯度。

In [ ]:
def num_grad_scalar(f, xs, eps=1e-6):
    '''f: 接受 float 列表、返回 float。对每个输入求中心差分。'''
    g = []
    for i in range(len(xs)):
        plus  = list(xs); plus[i]  += eps
        minus = list(xs); minus[i] -= eps
        g.append((f(plus) - f(minus)) / (2 * eps))
    return g

# 被测函数：g(x0,x1) = tanh(x0*x1) + x0**3
def expr_value(xs):
    x0, x1 = Value(xs[0]), Value(xs[1])
    out = (x0 * x1).tanh() + x0 ** 3
    backward(out)
    return out, [x0.grad, x1.grad]

def expr_float(xs):
    return math.tanh(xs[0]*xs[1]) + xs[0]**3

xs = [0.7, -1.3]
_, g_analytic = expr_value(xs)
g_numeric = num_grad_scalar(expr_float, xs)
for i, (ga, gn) in enumerate(zip(g_analytic, g_numeric)):
    print(f'  d/dx{i}: 解析={ga:+.6f}  数值={gn:+.6f}')
    assert abs(ga - gn) < 1e-5, f'输入 {i} 梯度不一致'
print('✅ 标量引擎与数值梯度对拍通过 —— autograd 正确')

## 4 · 张量版 autograd：VJP + 广播梯度

深度学习是张量运算。结构不变，只是每个算子的 VJP 变成张量运算，并要处理**广播**：
**前向广播了哪个维度，反向就沿那个维度 `sum` 回去**。先写关键工具 `unbroadcast`，再写一个张量 `Tensor` 类。

In [ ]:
def unbroadcast(grad, shape):
    '''把 grad 还原回 shape：对多出的前导维求和、对被广播(原size=1)的维求和。'''
    # (1) 维度数多出来的，从前面 sum 掉
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    # (2) 原本 size=1 但被广播成 >1 的维，sum 回去(保持维度)
    for ax, s in enumerate(shape):
        if s == 1 and grad.shape[ax] != 1:
            grad = grad.sum(axis=ax, keepdims=True)
    return grad.reshape(shape)

# 测试：(d,) 偏置被广播到 (n,d)，梯度应沿 n 求和回 (d,)
g_up = np.ones((4, 3))
assert unbroadcast(g_up, (3,)).shape == (3,)
assert np.allclose(unbroadcast(g_up, (3,)), [4, 4, 4])   # 每列 4 个 1 相加
assert unbroadcast(np.ones((4, 3)), (1, 3)).shape == (1, 3)
print('✅ unbroadcast 正确：广播=复制 -> 它的 VJP=沿被广播维求和')

In [ ]:
class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.asarray(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None
    @property
    def shape(self): return self.data.shape

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  = self.grad  + unbroadcast(out.grad, self.shape)
            other.grad = other.grad + unbroadcast(out.grad, other.shape)
        out._backward = _backward
        return out

    def __mul__(self, other):           # 逐元素
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  = self.grad  + unbroadcast(other.data * out.grad, self.shape)
            other.grad = other.grad + unbroadcast(self.data  * out.grad, other.shape)
        out._backward = _backward
        return out

    def matmul(self, other):
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            self.grad  = self.grad  + out.grad @ other.data.T   # X̄ = Ȳ Wᵀ
            other.grad = other.grad + self.data.T @ out.grad    # W̄ = Xᵀ Ȳ
        out._backward = _backward
        return out
    __matmul__ = matmul

    def relu(self):
        out = Tensor(np.maximum(self.data, 0), (self,), 'relu')
        def _backward():
            self.grad = self.grad + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), (self,), 'sum')
        def _backward():
            self.grad = self.grad + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

def backward_tensor(root):
    topo, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for ch in v._prev: build(ch)
            topo.append(v)
    build(root)
    root.grad = np.ones_like(root.data)   # 标量 loss -> 1
    for v in reversed(topo): v._backward()

# 端到端测试 + 数值梯度对拍：loss = sum( relu(X@W + b) )
Xv = rng.standard_normal((4, 3)); Wv = rng.standard_normal((3, 2)); bv = rng.standard_normal((2,))
X, W, b = Tensor(Xv), Tensor(Wv), Tensor(bv)
loss = (X @ W + b).relu().sum()
backward_tensor(loss)

def loss_of(Wn, bn):
    return np.maximum(Xv @ Wn + bn, 0).sum()
def numgrad(fn, P, eps=1e-6):
    g = np.zeros_like(P); it = np.nditer(P, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index; old = P[i]
        P[i] = old+eps; fp = fn()
        P[i] = old-eps; fm = fn()
        P[i] = old; g[i] = (fp-fm)/(2*eps); it.iternext()
    return g
gW = numgrad(lambda: loss_of(Wv, bv), Wv)
gb = numgrad(lambda: loss_of(Wv, bv), bv)
print('W.grad vs 数值  max|err| =', np.abs(W.grad - gW).max())
print('b.grad vs 数值  max|err| =', np.abs(b.grad - gb).max())
assert np.allclose(W.grad, gW, atol=1e-4)
assert np.allclose(b.grad, gb, atol=1e-4) and b.grad.shape == (2,)
print('✅ 张量 autograd 端到端对拍数值梯度通过；注意 b.grad 已沿 batch 维 sum 回 (2,)')

## 5 · 用自写 autograd 训练一个小 MLP

最有说服力的验证：用从零写的张量 autograd **真的训练一个网络**。
两层 MLP 拟合一个小回归任务，验证 (1) loss **单调下降**，(2) 训练后误差很小。

In [ ]:
# 造一个小回归任务：y = sin(线性组合)，用 MLP 拟合
n, din, dh = 64, 3, 16
Xtr = rng.standard_normal((n, din))
true_w = rng.standard_normal((din, 1))
ytr = np.tanh(Xtr @ true_w)                 # 目标 (n,1)

# 参数（小初始化）
W1 = Tensor(rng.standard_normal((din, dh)) * 0.3)
b1 = Tensor(np.zeros(dh))
W2 = Tensor(rng.standard_normal((dh, 1)) * 0.3)
b2 = Tensor(np.zeros(1))
params = [W1, b1, W2, b2]
Y = Tensor(ytr)

def forward():
    h = (Tensor(Xtr) @ W1 + b1).relu()
    pred = h @ W2 + b2
    diff = pred + (Y * -1)                   # pred - y
    return (diff * diff).sum()              # SSE

lr = 0.02
losses = []
for step in range(60):
    for pr in params: pr.grad = np.zeros_like(pr.data)   # zero_grad（因为 += 累加）
    loss = forward()
    backward_tensor(loss)
    for pr in params:
        pr.data = pr.data - lr * pr.grad / n             # 梯度下降（操作 .data）
    losses.append(float(loss.data) / n)

print(f'初始 MSE = {losses[0]:.4f}   最终 MSE = {losses[-1]:.4f}')
assert losses[-1] < losses[0] * 0.5, 'loss 应显著下降'
assert losses[-1] < losses[5], '后期应继续下降'
print('✅ 自写 autograd 把 MLP 训出来了！loss.backward() 真的没有魔法。')

---
## ✏️ 练习 1：给标量 `Value` 加 `exp` 算子

给 `Value` 类补一个 `exp()` 方法：前向 `out.data = exp(x)`，反向 `x.grad += exp(x) * out.grad`（即 `out.data * out.grad`）。

实现 `value_exp(v)`（返回新 `Value`，并正确挂 `_backward`），然后用它和已有算子搭一个表达式，对拍数值梯度。

In [ ]:
def value_exp(v):
    # TODO: 返回 out = Value(exp(v.data), (v,), 'exp')
    #       并设 out._backward 使 v.grad += out.data * out.grad
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# f(x) = exp(x) * x ; df/dx = exp(x)*(x+1)
x = Value(0.5)
f = value_exp(x) * x
backward(f)
expected = math.exp(0.5) * (0.5 + 1)
assert abs(x.grad - expected) < 1e-6, f'得到 {x.grad}, 期望 {expected}'
# 再和数值梯度对拍
def f_float(xs): return math.exp(xs[0]) * xs[0]
gn = num_grad_scalar(f_float, [0.5])[0]
assert abs(x.grad - gn) < 1e-5
print(f'✅ 练习 1 通过：exp 算子的 VJP 正确 (grad={x.grad:.4f})')

## ✏️ 练习 2：广播梯度——给 `Tensor` 实现 `mean`

给 `Tensor` 实现 `tmean(t)`：前向 `out = t.data.mean()`（标量），反向每个元素的梯度是 `out.grad / N`（N=元素个数）。
这考你「前向把 N 个数聚合成 1 个，反向要把梯度均摊回 N 个元素」。

In [ ]:
def tmean(t):
    # TODO: out = Tensor(t.data.mean(), (t,), 'mean')
    #       _backward: t.grad += ones_like(t.data) * out.grad / t.data.size
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Av = rng.standard_normal((3, 4))
A = Tensor(Av)
m = tmean(A)
backward_tensor(m)
assert m.data.shape == ()           # 标量
assert np.allclose(A.grad, np.ones((3,4)) / 12), 'mean 的梯度应是 1/N 均摊'
# 对拍数值梯度
g_num = numgrad(lambda: Av.mean(), Av)
assert np.allclose(A.grad, g_num, atol=1e-6)
print('✅ 练习 2 通过：mean 的反向把梯度 1/N 均摊回每个元素')

## ✏️ 练习 3：自定义算子——straight-through `round`

实现一个**直通估计（straight-through estimator）**算子 `ste_round(t)`：
- **前向**：`out.data = round(t.data)`（不可微的取整）；
- **反向**：假装它是恒等函数，梯度**直接穿过** `t.grad += out.grad`（无视 round 的真实导数 0）。

这正是量化训练里让梯度流过取整的标准技巧，靠的是 autograd 把 forward/backward 解耦。

In [ ]:
def ste_round(t):
    # TODO: out = Tensor(np.round(t.data), (t,), 'ste_round')
    #       _backward: t.grad += out.grad   (直通，无视 round 的导数)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
v = Tensor(np.array([0.2, 0.8, 1.4, -0.3]))
out = ste_round(v)
loss = out.sum()
backward_tensor(loss)
assert np.allclose(out.data, [0., 1., 1., 0.]), '前向应是 round'
assert np.allclose(v.grad, [1., 1., 1., 1.]), '反向应直通(梯度=1)，而非 round 的真实导数 0'
print('✅ 练习 3 通过：STE 让梯度直接穿过不可微的 round —— forward/backward 解耦的威力')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def value_exp(v):
    out = Value(math.exp(v.data), (v,), 'exp')
    def _backward():
        v.grad += out.data * out.grad      # d exp(x)/dx = exp(x) = out.data
    out._backward = _backward
    return out

In [ ]:
# 练习 2 参考答案
def tmean(t):
    out = Tensor(t.data.mean(), (t,), 'mean')
    def _backward():
        t.grad = t.grad + np.ones_like(t.data) * out.grad / t.data.size
    out._backward = _backward
    return out

In [ ]:
# 练习 3 参考答案
def ste_round(t):
    out = Tensor(np.round(t.data), (t,), 'ste_round')
    def _backward():
        t.grad = t.grad + out.grad          # 直通：假装恒等
    out._backward = _backward
    return out

---
## 🧪 真实数据胶囊：与真实 PyTorch autograd 对拍

我们手写的 autograd，应当与 **真实 `torch.autograd`** 给出**相同**的梯度。
下面在同一个表达式上，让我们的 `Tensor` 引擎与 PyTorch 各算一遍并对拍。

**装了 torch 才会实跑对拍；没装则用预先存好的真实数值对拍（不阻断）。** 这就是本课「真实 API 对照 + 优雅回退」的范式。

In [ ]:
# 同一个表达式：loss = sum( relu(X@W + b) * 2 )
Xc = np.array([[ 1.0, -2.0,  0.5],
               [-1.0,  0.3,  2.0]])
Wc = np.array([[ 0.5,  1.0],
               [-1.0,  0.2],
               [ 0.3, -0.4]])
bc = np.array([0.1, -0.2])

# 我们的引擎
Xt, Wt, bt = Tensor(Xc), Tensor(Wc), Tensor(bc)
loss = ((Xt @ Wt + bt).relu() * 2).sum()
backward_tensor(loss)
ours_W, ours_b = Wt.grad.copy(), bt.grad.copy()
print('我们的引擎 loss =', round(float(loss.data), 6))
print('我们的 W.grad =\n', ours_W)

**🧪 胶囊练习**：实现 `torch_or_reference_grads()`：
- 若有 torch：用 `torch.tensor(..., requires_grad=True)` 复算同一 `loss` 并 `.backward()`，返回 `(W.grad, b.grad)` 的 numpy；
- 若没 torch：返回预先用 torch 算好、存在 `REF_W`/`REF_B` 里的真实数值。

然后对拍我们的引擎 == torch（或参考值）。学生骨架（不计入自动验证）：

In [ ]:
# 预先用真实 PyTorch 算好的参考梯度（loss = sum(relu(Xc@Wc+bc)*2)）
# pre-activation = [[2.75, 0.2],[-0.1,-1.94]]，relu 保留前两个、屏蔽后两个
REF_W = np.array([[ 2.0,  2.0],
                  [-4.0, -4.0],
                  [ 1.0,  1.0]])     # Xcᵀ @ (2 * relu_mask)
REF_B = np.array([2.0, 2.0])          # 每列存活样本数 * 2

def torch_or_reference_grads():
    # TODO: 有 torch 则实算并返回 (Wgrad, bgrad)；否则返回 (REF_W, REF_B)
    raise NotImplementedError

In [ ]:
# 自测（学生填好上面后运行）
ref_W, ref_B = torch_or_reference_grads()
assert np.allclose(ours_W, ref_W, atol=1e-5), '我们的 W.grad 应与 torch/参考一致'
assert np.allclose(ours_b, ref_B, atol=1e-5), '我们的 b.grad 应与 torch/参考一致'
print('✅ 胶囊通过：自写 autograd 与真实 torch.autograd 给出相同梯度')

In [ ]:
# 📖 胶囊参考答案
def torch_or_reference_grads():
    try:
        import torch
        X = torch.tensor(Xc, requires_grad=False)
        W = torch.tensor(Wc, requires_grad=True)
        b = torch.tensor(bc, requires_grad=True)
        loss = (torch.relu(X @ W + b) * 2).sum()
        loss.backward()
        return W.grad.numpy(), b.grad.numpy()
    except Exception:
        return REF_W, REF_B

> 说明：上面 `REF_W/REF_B` 是用真实 PyTorch 跑同一表达式得到的梯度。两个样本的 pre-activation 是 `[[2.75,0.2],[-0.1,-1.94]]`，relu 只保留第一行（两个正值）、屏蔽第二行（两个负值），`*2` 使存活项梯度翻倍。
无论有没有 torch，**自写引擎都与之逐位一致**——这就是「numpy 复刻 + 真实框架对照」的闭环。

---
## 🔧 旁注：真实 PyTorch / 自定义 Function 长什么样

我们手写的 `Value`/`Tensor` + `backward`，在 PyTorch 里就是内建的 autograd（伪代码/对照，**不依赖即可读**）：

```python
import torch
x = torch.randn(4, 3, requires_grad=True)
W = torch.randn(3, 2, requires_grad=True)
b = torch.randn(2, requires_grad=True)
loss = torch.relu(x @ W + b).sum()
loss.backward()           # == 我们的 backward_tensor(loss)
x.grad, W.grad, b.grad    # 自动填好；b.grad 已沿 batch 维 sum 回 (2,)
```

我们的 `_backward` 闭包 ↔ PyTorch 的 `grad_fn`（看 `loss.grad_fn`）；我们的拓扑逆序 ↔ PyTorch 的反向引擎。

自定义算子（练习 3 的 STE）在 PyTorch 里是 `torch.autograd.Function`：

```python
class STERound(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):  return torch.round(x)
    @staticmethod
    def backward(ctx, g): return g          # 直通，无视 round 的导数
```

`forward`/`backward` 的契约，与我们给每个算子写 `data` + `_backward` 完全同构。

### 小结
- autograd = **计算图（谁由谁算来）+ 每算子 VJP 规则 + 拓扑逆序遍历**。没有魔法。
- 深度学习用**反向模式**（一个标量 loss 对所有参数，一次反向搞定）；原子操作是 **VJP**（不显式构造雅可比）。
- 头号 bug 是**梯度累加**：分叉节点的梯度要 `+=` 求和；故每步前 `zero_grad`。
- 张量版唯一新增复杂度是**广播梯度**：前向广播哪维，反向沿那维 `sum` 回去（`unbroadcast`）。
- 自定义算子 = 往「VJP 规则集」里加一条；STE / `detach` 都靠 forward/backward **解耦**。

下一站：**模块 02 · torch.compile 图捕获与编译** —— 有了 autograd，怎么把整段计算捕获成图、融合、编译得更快。